# Chess RL vs Stockfish - Clean Implementation
Greedy move selection, proper loss calculation, detailed logging

In [ ]:
!apt-get update && apt-get install -y stockfish -qq
!pip install python-chess torch numpy matplotlib -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import chess
import chess.engine
import chess.pgn
import numpy as np
import json
from pathlib import Path
import random
import matplotlib.pyplot as plt
from datetime import datetime

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = Path('/content/drive/MyDrive/chess_final')
drive_path.mkdir(parents=True, exist_ok=True)
print(f'Save path: {drive_path}')

## Board Encoder (Simple 13-plane)

In [ ]:
class BoardEncoder:
    def encode(self, board):
        tensor = torch.zeros(13, 8, 8, dtype=torch.float32)
        for square in range(64):
            piece = board.piece_at(square)
            if piece:
                row, col = square // 8, square % 8
                piece_idx = piece.piece_type - 1
                plane = piece_idx if piece.color == chess.WHITE else piece_idx + 6
                tensor[plane, row, col] = 1.0
        if board.turn == chess.BLACK:
            tensor[12, :, :] = 1.0
        return tensor

encoder = BoardEncoder()
test = encoder.encode(chess.Board())
print(f'Encoder shape: {test.shape}')

## Network (Simple CNN)

In [ ]:
class ChessNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(13, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.conv3 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        self.value_fc1 = nn.Linear(128 * 64, 256)
        self.value_fc2 = nn.Linear(256, 1)
        
        self.policy_fc1 = nn.Linear(128 * 64, 256)
        self.policy_fc2 = nn.Linear(256, 1)
    
    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        x = torch.relu(self.bn2(self.conv2(x)))
        x = torch.relu(self.bn3(self.conv3(x)))
        x_flat = x.view(x.size(0), -1)
        
        value = torch.tanh(self.value_fc2(torch.relu(self.value_fc1(x_flat))))
        policy = self.policy_fc2(torch.relu(self.policy_fc1(x_flat)))
        
        return value, policy

net = ChessNet().to(device)
params = sum(p.numel() for p in net.parameters())
print(f'Network: {params:,} parameters')

## Greedy Move Selection

In [ ]:
def select_best_move(board, network, encoder):
    legal_moves = list(board.legal_moves)
    if not legal_moves:
        return None, 0.0
    
    best_move = None
    best_score = -float('inf')
    
    for move in legal_moves:
        board.push(move)
        tensor = encoder.encode(board).unsqueeze(0).to(device)
        
        with torch.no_grad():
            value, _ = network(tensor)
            score = value.item()
        
        board.pop()
        
        if score > best_score:
            best_score = score
            best_move = move
    
    return best_move, best_score

print('Move selector ready')

## Stockfish Engine

In [ ]:
class StockfishBot:
    def __init__(self, depth=1):
        self.engine = chess.engine.SimpleEngine.popen_uci('/usr/games/stockfish')
        self.depth = depth
    
    def get_move(self, board):
        try:
            result = self.engine.play(board, chess.engine.Limit(depth=self.depth))
            return result.move
        except Exception as e:
            print(f'Engine error: {e}')
            return random.choice(list(board.legal_moves))
    
    def set_depth(self, depth):
        print(f'  Stockfish depth: {self.depth} -> {depth}')
        self.depth = depth
    
    def quit(self):
        try:
            self.engine.quit()
        except:
            pass

print('Stockfish wrapper ready')

## Game Loop

In [ ]:
def play_game(network, encoder, stockfish, nn_is_white=True, max_moves=200):
    board = chess.Board()
    moves_data = []
    move_list = []
    
    for move_count in range(max_moves):
        if board.is_game_over():
            result = board.result()
            if result == '1-0':
                winner = 0 if nn_is_white else 1
            elif result == '0-1':
                winner = 1 if nn_is_white else 0
            else:
                winner = 2
            return winner, moves_data, len(move_list)
        
        is_nn_turn = (board.turn == chess.WHITE) if nn_is_white else (board.turn == chess.BLACK)
        
        if is_nn_turn:
            move, score = select_best_move(board, network, encoder)
            if move is None:
                break
        else:
            move = stockfish.get_move(board)
            if move is None:
                break
            score = 0.0
        
        board_before = encoder.encode(board).clone()
        board.push(move)
        move_list.append(move)
        
        if is_nn_turn:
            moves_data.append({
                'board': board_before,
                'move': move,
                'nn_score': score
            })
    
    return 2, moves_data, len(move_list)

print('Game loop ready')

## Training with Proper Loss

In [ ]:
def train_on_game(network, optimizer, moves_data, winner, learning_rate=0.001):
    if not moves_data:
        return 0.0, 0.0, 0.0
    
    network.train()
    total_value_loss = 0.0
    total_policy_loss = 0.0
    batch_count = 0
    batch_size = 8
    
    for i in range(0, len(moves_data), batch_size):
        batch_moves = moves_data[i:i+batch_size]
        boards_tensor = torch.stack([m['board'] for m in batch_moves]).to(device)
        
        values, policies = network(boards_tensor)
        
        if winner == 0 or winner == 1:
            target_value = torch.tensor(1.0, dtype=torch.float32).to(device)
            value_weight = 0.3
        elif winner == 2:
            target_value = torch.tensor(-0.5, dtype=torch.float32).to(device)
            value_weight = 2.0
        else:
            target_value = torch.tensor(-1.0, dtype=torch.float32).to(device)
            value_weight = 1.5
        
        target_values = target_value.expand(len(batch_moves), 1)
        
        value_loss = nn.MSELoss()(values, target_values)
        value_loss = value_loss * value_weight
        
        policy_loss = -torch.mean(policies)
        
        total_loss = value_loss + policy_loss * 0.1
        
        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(network.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_value_loss += value_loss.item()
        total_policy_loss += policy_loss.item()
        batch_count += 1
    
    network.eval()
    
    avg_value_loss = total_value_loss / batch_count if batch_count > 0 else 0.0
    avg_policy_loss = total_policy_loss / batch_count if batch_count > 0 else 0.0
    avg_total_loss = avg_value_loss + avg_policy_loss * 0.1
    
    return avg_total_loss, avg_value_loss, avg_policy_loss

print('Training ready')

## Save/Load Checkpoints

In [ ]:
def save_checkpoint(network, optimizer, episode, depth, drive_path):
    checkpoint = {
        'episode': episode,
        'depth': depth,
        'model_state': network.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'timestamp': datetime.now().isoformat()
    }
    path = drive_path / f'checkpoint_ep{episode}_d{depth}.pt'
    torch.save(checkpoint, path)
    return path

def load_checkpoint(path):
    checkpoint = torch.load(path, map_location=device)
    network = ChessNet().to(device)
    optimizer = optim.Adam(network.parameters(), lr=0.001)
    network.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    return network, optimizer, checkpoint['episode'], checkpoint['depth']

print('Checkpoint ready')

## Statistics Tracker

In [ ]:
class Stats:
    def __init__(self, drive_path):
        self.drive_path = drive_path
        self.file = drive_path / 'curriculum_stats.json'
        self.stats = {
            'total_games': 0,
            'current_depth': 1,
            'depths': {},
            'win_count': 0,
            'loss_count': 0,
            'draw_count': 0,
            'history': [],
            'loss_history': []
        }
        if self.file.exists():
            with open(self.file) as f:
                self.stats = json.load(f)
    
    def record(self, winner, depth, loss, value_loss, policy_loss, game_length):
        self.stats['total_games'] += 1
        self.stats['loss_history'].append({
            'total': loss,
            'value': value_loss,
            'policy': policy_loss
        })
        
        depth_str = str(depth)
        if depth_str not in self.stats['depths']:
            self.stats['depths'][depth_str] = {
                'wins': 0, 'losses': 0, 'draws': 0, 'games': 0, 'lengths': []
            }
        
        if winner == 0 or winner == 1:
            self.stats['win_count'] += 1
            self.stats['depths'][depth_str]['wins'] += 1
            self.stats['history'].append('W')
        elif winner == 2:
            self.stats['draw_count'] += 1
            self.stats['depths'][depth_str]['draws'] += 1
            self.stats['history'].append('D')
        else:
            self.stats['loss_count'] += 1
            self.stats['depths'][depth_str]['losses'] += 1
            self.stats['history'].append('L')
        
        self.stats['depths'][depth_str]['games'] += 1
        self.stats['depths'][depth_str]['lengths'].append(game_length)
        self.stats['history'] = self.stats['history'][-100:]
    
    def get_win_rate(self, depth):
        depth_str = str(depth)
        if depth_str not in self.stats['depths']:
            return 0.0
        d = self.stats['depths'][depth_str]
        total = d['wins'] + d['losses'] + d['draws']
        return d['wins'] / total if total > 0 else 0.0
    
    def save(self):
        with open(self.file, 'w') as f:
            json.dump(self.stats, f, indent=2)
    
    def print_depth_stats(self, depth):
        depth_str = str(depth)
        if depth_str not in self.stats['depths']:
            return
        d = self.stats['depths'][depth_str]
        total = d['wins'] + d['losses'] + d['draws']
        if total == 0:
            return
        
        avg_loss = np.mean([l['total'] for l in self.stats['loss_history'][-20:]]) if self.stats['loss_history'] else 0
        avg_len = np.mean(d['lengths'][-20:]) if d['lengths'] else 0
        wr = d['wins'] / total * 100
        
        print(f"  Depth {depth} | W:{d['wins']:2d} ({wr:5.1f}%) L:{d['losses']:2d} D:{d['draws']:2d} | Loss:{avg_loss:.6f} | AvgLen:{avg_len:.0f}")

stats = Stats(drive_path)
print('Stats ready')

## Main Training Loop

In [ ]:
WIN_THRESHOLD = 0.60
GAMES_PER_DEPTH = 50
LEARNING_RATE = 0.001
START_DEPTH = 1
MAX_DEPTH = 20

print("\n" + "="*80)
print("CHESS RL vs STOCKFISH - CURRICULUM LEARNING")
print("="*80)
print(f"Win threshold: {WIN_THRESHOLD*100:.0f}%")
print(f"Games per depth: {GAMES_PER_DEPTH}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Start depth: {START_DEPTH}")
print("="*80 + "\n")

checkpoints = sorted(drive_path.glob('checkpoint_*.pt'))
start_episode = 0
current_depth = START_DEPTH

if checkpoints:
    print(f"Loading checkpoint: {checkpoints[-1].name}")
    net, optimizer, start_episode, current_depth = load_checkpoint(checkpoints[-1])
    print(f"Resumed from episode {start_episode}, depth {current_depth}")
    stats.print_depth_stats(current_depth)
    print()
else:
    net = ChessNet().to(device)
    optimizer = optim.Adam(net.parameters(), lr=LEARNING_RATE)
    net.eval()
    print("Starting fresh training\n")

stockfish = StockfishBot(depth=current_depth)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)

episode = start_episode

try:
    while episode < start_episode + 500:
        nn_is_white = (episode % 2 == 0)
        winner, moves_data, game_length = play_game(net, encoder, stockfish, nn_is_white=nn_is_white)
        
        loss, value_loss, policy_loss = train_on_game(net, optimizer, moves_data, winner, LEARNING_RATE)
        stats.record(winner, current_depth, loss, value_loss, policy_loss, game_length)
        
        episode += 1
        
        if episode % 10 == 0:
            result_char = 'W' if winner in [0, 1] else ('D' if winner == 2 else 'L')
            print(f"Episode {episode:4d} | Result: {result_char} | Len: {game_length:3d} | Loss: {loss:.6f} (V:{value_loss:.6f} P:{policy_loss:.6f}) | ", end='')
            stats.print_depth_stats(current_depth)
        
        games_at_depth = stats.stats['depths'].get(str(current_depth), {}).get('games', 0)
        if games_at_depth >= GAMES_PER_DEPTH:
            wr = stats.get_win_rate(current_depth)
            if wr >= WIN_THRESHOLD and current_depth < MAX_DEPTH:
                new_depth = current_depth + 1
                stockfish.set_depth(new_depth)
                current_depth = new_depth
                print(f"\n🚀 LEVEL UP! Depth {new_depth-1} -> {new_depth} (Win rate: {wr*100:.1f}%)\n")
        
        if episode % 50 == 0:
            save_checkpoint(net, optimizer, episode, current_depth, drive_path)
            stats.save()
            print(f"\n💾 Checkpoint saved at episode {episode}\n")
        
        scheduler.step()

except KeyboardInterrupt:
    print(f"\n\nInterrupted at episode {episode}")
    save_checkpoint(net, optimizer, episode, current_depth, drive_path)
    stats.save()
    print("Checkpoint saved\n")

finally:
    stockfish.quit()
    print("Stockfish engine closed")

print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80)
print(f"Total games: {stats.stats['total_games']}")
print(f"Wins: {stats.stats['win_count']} | Losses: {stats.stats['loss_count']} | Draws: {stats.stats['draw_count']}")
print(f"Final depth: {current_depth}")
print("="*80)

## Visualization

In [ ]:
with open(drive_path / 'curriculum_stats.json') as f:
    s = json.load(f)

if s['total_games'] > 10:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    ax = axes[0, 0]
    ax.pie([s['win_count'], s['loss_count'], s['draw_count']],
           labels=['Wins', 'Losses', 'Draws'],
           autopct='%1.1f%%',
           colors=['#2ecc71', '#e74c3c', '#95a5a6'])
    ax.set_title(f"Overall Results ({s['total_games']} games)")
    
    ax = axes[0, 1]
    if s['loss_history']:
        losses = [l['total'] for l in s['loss_history']]
        ax.plot(losses, alpha=0.7, linewidth=1)
        ax.set_title('Training Loss (Total)')
        ax.set_xlabel('Episode')
        ax.set_ylabel('Loss')
        ax.grid(True, alpha=0.3)
    
    ax = axes[1, 0]
    depths = sorted([int(d) for d in s['depths'].keys()])
    win_rates = []
    for d in depths:
        d_str = str(d)
        g = s['depths'][d_str]
        total = g['wins'] + g['losses'] + g['draws']
        wr = g['wins'] / total if total > 0 else 0
        win_rates.append(wr)
    
    ax.plot(depths, win_rates, marker='o', linewidth=2, markersize=8, color='#3498db')
    ax.axhline(y=0.6, color='green', linestyle='--', alpha=0.5, label='Level-up threshold')
    ax.set_xlabel('Stockfish Depth')
    ax.set_ylabel('Win Rate')
    ax.set_title('Win Rate by Depth')
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])
    ax.legend()
    
    ax = axes[1, 1]
    game_counts = []
    for d in depths:
        d_str = str(d)
        game_counts.append(s['depths'][d_str]['games'])
    
    ax.bar(depths, game_counts, color='#9b59b6', alpha=0.7)
    ax.set_xlabel('Stockfish Depth')
    ax.set_ylabel('Games Played')
    ax.set_title('Games per Depth Level')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(drive_path / 'curriculum_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved curriculum_results.png')
else:
    print('Not enough games yet')